## 3. Knowledge Base Preparation

### Importing Libraries

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

### 1. Loading the document

In [4]:
file_path = '../Knowledge_Bases/Brightloop_Company_Policies_and_Business_Rules.txt'

In [6]:
loader = TextLoader(file_path)
data = loader.load()
print(data[0].page_content)

# BRIGHTLOOP SOFTWARE — COMPANY POLICIES AND BUSINESS RULES KNOWLEDGE BASE

## 1. DOCUMENT OVERVIEW AND GOVERNANCE

### Document Purpose

This document is the official statement of how Brightloop Software runs its subscription business. It defines who may perform each action, which conditions must be satisfied, and what happens when circumstances change.

The Staxio subscription management platform is built to implement these policies. When the system and this document differ, this document governs and the system must be corrected.

### Scope of the Policy

The policy covers the whole life of a subscription: setting up products, plans, taxes and discounts; preparing quotations; activating subscriptions; generating invoices; recording payments; chasing overdue amounts; customer self-service; and revenue reporting.

### Policy Governance

The Administrator owns this policy and is the only person who may approve a revision. The Finance Lead reviews any change that affects revenue or invoi

### 2. Chunking the Document

In [7]:
headers_to_split_on = [
    ("#", "section"),
    ("##", "subsection"),
    ("###", "topic"),
]

In [8]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True
)

In [9]:
chunks = splitter.split_text(data[0].page_content)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 205


In [10]:
for i, chunk in enumerate(chunks):
    print(f"\n{'=' * 20} CHUNK {i} {'=' * 20}")
    print(chunk.page_content)


==================== CHUNK 0 ====================
This document is the official statement of how Brightloop Software runs its subscription business. It defines who may perform each action, which conditions must be satisfied, and what happens when circumstances change.  
The Staxio subscription management platform is built to implement these policies. When the system and this document differ, this document governs and the system must be corrected.

==================== CHUNK 1 ====================
The policy covers the whole life of a subscription: setting up products, plans, taxes and discounts; preparing quotations; activating subscriptions; generating invoices; recording payments; chasing overdue amounts; customer self-service; and revenue reporting.

==================== CHUNK 2 ====================
The Administrator owns this policy and is the only person who may approve a revision. The Finance Lead reviews any change that affects revenue or invoices before approval.  
Changes to 

### 3. Embedding the Document

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

/Users/tanmaykhomane/Artificial Intelligence/GenAI IBM/Projects/team_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
document_vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks])

print(f"Number of vectors: {len(document_vectors)}")
print(f"Vector Size: {len(document_vectors[0])}")

Number of vectors: 205
Vector Size: 384


### 4. Store the Vectors in ChromaDB

In [14]:
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    collection_name = "StaxioAI_KB",
    persist_directory="../../vector_store/chroma_db"
)

In [15]:
query = "How are customer payments recorded and validated?"

similarity_results = vector_store.similarity_search(
    query,
    k=4
)

In [16]:
for i, doc in enumerate(similarity_results, start=1):
    print(f"\n--- Result {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content)


--- Result 1 ---
Metadata: {'subsection': '17. PAYMENT MANAGEMENT', 'topic': 'Payment Recording', 'section': 'CATEGORY 4: BILLING, INVOICES AND PAYMENTS'}
Administrators and Staff record payments. Finance Lead oversees payment recording and reviews payment records. Customers cannot record payments.  
Each payment is recorded against one invoice with its method, amount, payment date and a note.  
Each payment gets a unique number, PAY- followed by the five-digit record number. The system keeps who recorded it and when. (Database)  
For bank transfer, UPI and cheque payments, the note shall contain the bank or transaction reference or the cheque number.

--- Result 2 ---
Metadata: {'subsection': '17. PAYMENT MANAGEMENT', 'topic': 'Payment Reconciliation', 'section': 'CATEGORY 4: BILLING, INVOICES AND PAYMENTS'}
Staff can search payments by customer, invoice, date, method and amount to verify them against bank statements.  
Finance Lead is responsible for reviewing payment reconciliation